# Fine-tuning DomURLs-BERT com Tokens Especiais

**Objetivo:** Re-treinar o DomURLs-BERT adicionando tokens especiais (`[URL]`, `[WHOIS]`, `[EXTRA]`, etc.) ao vocabulário do tokenizer antes do fine-tuning, para que o modelo aprenda a usar features multimodais (URL + WHOIS + features numéricas).

**Problema original:** O tokenizer base do DomURLs-BERT não conhece esses tokens — mapeava todos para `[UNK]`, corrompendo as predições.

**Solução:** `tokenizer.add_special_tokens()` + `model.resize_token_embeddings()` antes do treino.

**Ambiente:** Google Colab T4 (GPU gratuita)

**Dataset:** `dataset_multimodal.csv` (URL + WHOIS + features DOM)

In [ ]:
# ============================================================
# 1. Verificação de GPU e Instalação
# ============================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('AVISO: Sem GPU! O treino será muito lento.')
    print('Vá em Runtime > Change runtime type > T4 GPU')
print(f'Device: {device}')

!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
import os, time, gc, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report, average_precision_score, log_loss
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
# ============================================================
# 3. Configuração
# ============================================================
MODEL_ID          = 'amahdaouy/DomURLs_BERT'   # modelo base
RANDOM_STATE      = 42
MAX_LENGTH        = 192      # tokens (> 128 p/ acomodar WHOIS + features)
NUM_EPOCHS        = 3
BATCH_SIZE        = 32       # T4 16GB aguenta 32 com fp16
GRAD_ACCUM        = 2        # efetivo = 64
LR                = 2e-5
WARMUP_RATIO      = 0.06
WEIGHT_DECAY      = 0.01
MAX_TRAIN_SAMPLES = 200_000  # None = tudo
MAX_EVAL_SAMPLES  = 50_000   # None = tudo

# Tokens especiais que o DomURLs-BERT original NÃO conhece
SPECIAL_TOKENS = ['[URL]', '[WHOIS]', '[EXTRA]', '[AGE]', '[REG]', '[EXPIRE]']

# Google Drive
DRIVE_BASE = '/content/drive/MyDrive/TCC-Finetuning-DomURLs-BERT'
OUTPUT_DIR = f'{DRIVE_BASE}/modelo-final'

print(f'Modelo base:  {MODEL_ID}')
print(f'Max length:   {MAX_LENGTH}')
print(f'Batch efetivo: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Tokens novos: {SPECIAL_TOKENS}')

In [ ]:
# ============================================================
# 4. Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Drive montado. Output: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# 5. Carregar Dataset
# ============================================================
# Faça upload do dataset_multimodal.csv para o Colab ou Drive
DATASET_PATH = '/content/dataset_multimodal.csv'
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = f'{DRIVE_BASE}/dataset_multimodal.csv'
if not os.path.exists(DATASET_PATH):
    from google.colab import files
    print('Faça upload do dataset_multimodal.csv:')
    uploaded = files.upload()
    DATASET_PATH = '/content/dataset_multimodal.csv'

df = pd.read_csv(DATASET_PATH, encoding='utf-8')
print(f'Dataset: {len(df):,} amostras')
print(f'Colunas: {list(df.columns)}')
print(f'Phishing: {df["label"].mean():.1%}')
print(f'Fontes: {df["source"].value_counts().to_dict()}')
print(f'\nExemplo:')
print(f'  {df.iloc[0]["text_input"][:200]}')

In [ ]:
# ============================================================
# 6. Split Treino / Avaliação
# ============================================================
train_df, eval_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=RANDOM_STATE
)

# Amostragem estratificada se necessário
if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    frac = MAX_TRAIN_SAMPLES / len(train_df)
    train_df = (
        train_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

if MAX_EVAL_SAMPLES and len(eval_df) > MAX_EVAL_SAMPLES:
    frac = MAX_EVAL_SAMPLES / len(eval_df)
    eval_df = (
        eval_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

train_texts  = train_df['text_input'].tolist()
eval_texts   = eval_df['text_input'].tolist()
train_labels = train_df['label'].astype(int).tolist()
eval_labels  = eval_df['label'].astype(int).tolist()

print(f'Treino:    {len(train_texts):,}')
print(f'Avaliação: {len(eval_texts):,}')
print(f'Phishing treino: {sum(train_labels)/len(train_labels):.1%}')
print(f'Phishing eval:   {sum(eval_labels)/len(eval_labels):.1%}')

In [ ]:
# ============================================================
# 7. Carregar Tokenizer + Adicionar Tokens Especiais
# ============================================================
# Esta é a correção principal: adicionar os tokens ao vocabulário
# ANTES do treino, para que o modelo aprenda seus embeddings.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

print(f'Vocab ANTES: {len(tokenizer):,}')

# Verificar quais tokens já existem
unk_id = tokenizer.convert_tokens_to_ids('[UNK]')
for token in SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(token)
    status = 'EXISTE' if tid != unk_id else 'NÃO EXISTE (UNK)'
    print(f'  {token:10s} -> ID={tid} ({status})')

# Adicionar tokens que faltam
num_added = tokenizer.add_special_tokens({
    'additional_special_tokens': SPECIAL_TOKENS
})
print(f'\nTokens adicionados: {num_added}')
print(f'Vocab DEPOIS: {len(tokenizer):,}')

# Confirmar
for token in SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(token)
    print(f'  {token:10s} -> ID={tid}')

In [ ]:
# ============================================================
# 8. Carregar Modelo + Redimensionar Embeddings
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: 'legitimate', 1: 'phishing'},
    label2id={'legitimate': 0, 'phishing': 1},
    ignore_mismatched_sizes=True,
)

# Redimensionar embeddings para acomodar os novos tokens
old_size = model.config.vocab_size
model.resize_token_embeddings(len(tokenizer))
new_size = model.config.vocab_size

print(f'Embeddings: {old_size:,} -> {new_size:,} (+{new_size - old_size})')

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros totais:    {n_params/1e6:.1f}M')
print(f'Parâmetros treináveis: {n_train/1e6:.1f}M')

model.to(device)
print(f'Modelo em: {device}')

In [ ]:
# ============================================================
# 9. Verificação: tokenização de exemplo
# ============================================================
# Confirmar que os tokens especiais agora são reconhecidos
sample = '[URL] https://www.google.com [WHOIS] unknown [EXTRA] length=24 tls=1'
tokens = tokenizer.tokenize(sample)
print(f'Input:  {sample}')
print(f'Tokens: {tokens[:30]}...')
print(f'Total tokens: {len(tokens)}')
print()

# Verificar que [URL], [WHOIS], [EXTRA] aparecem como tokens únicos
for special in SPECIAL_TOKENS:
    if special.lower() in [t.lower() for t in tokens]:
        print(f'  {special} -> token único ✓')
    else:
        # Pode estar em lowercase no vocab
        found = [t for t in tokens if special.strip('[]').lower() in t.lower()]
        print(f'  {special} -> subword: {found}')

In [ ]:
# ============================================================
# 10. Dataset PyTorch
# ============================================================
class PhishingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

print('Tokenizando treino...')
train_ds = PhishingDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
print('Tokenizando avaliação...')
eval_ds  = PhishingDataset(eval_texts, eval_labels, tokenizer, MAX_LENGTH)
print(f'Train: {len(train_ds):,} | Eval: {len(eval_ds):,}')

In [ ]:
# ============================================================
# 11. Métricas
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    proba = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        'accuracy':  float(accuracy_score(labels, preds)),
        'f1':        float(f1_score(labels, preds, zero_division=0)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall':    float(recall_score(labels, preds, zero_division=0)),
        'mcc':       float(matthews_corrcoef(labels, preds)),
        'auc_roc':   float(roc_auc_score(labels, proba)),
    }

In [ ]:
# ============================================================
# 12. Treinamento
# ============================================================
training_args = TrainingArguments(
    output_dir='/content/tmp_finetuning',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=200,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=RANDOM_STATE,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

print(f'Iniciando treino: {NUM_EPOCHS} épocas, batch efetivo={BATCH_SIZE*GRAD_ACCUM}')
print(f'Steps por época: ~{len(train_ds) // (BATCH_SIZE * GRAD_ACCUM):,}')
print()

t_start = time.time()
trainer.train()
train_time = time.time() - t_start

print(f'\nTreino concluído em {train_time/60:.1f} minutos')

In [ ]:
# ============================================================
# 13. Avaliação Completa
# ============================================================
print('Gerando predições no eval set...')
model.eval()

eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

all_logits = []
all_labels = []
latencies  = []

with torch.no_grad():
    for batch in eval_loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        labels = batch['labels']

        t0 = time.perf_counter()
        outputs = model(**inputs)
        t1 = time.perf_counter()

        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels)
        latencies.append((t1 - t0) / len(labels) * 1000)  # ms per sample

logits = torch.cat(all_logits)
y_true = torch.cat(all_labels).numpy()
probs  = torch.softmax(logits, dim=-1).numpy()
y_pred = np.argmax(probs, axis=-1)
y_proba = probs[:, 1]

print(f'\n{"="*55}')
print(f'  RESULTADOS — DomURLs-BERT Fine-tuned (multimodal)')
print(f'{"="*55}')
print(f'  Accuracy:      {accuracy_score(y_true, y_pred):.4f}')
print(f'  F1-Score:      {f1_score(y_true, y_pred):.4f}')
print(f'  Precision:     {precision_score(y_true, y_pred):.4f}')
print(f'  Recall:        {recall_score(y_true, y_pred):.4f}')
print(f'  AUC-ROC:       {roc_auc_score(y_true, y_proba):.4f}')
print(f'  PR-AUC:        {average_precision_score(y_true, y_proba):.4f}')
print(f'  MCC:           {matthews_corrcoef(y_true, y_pred):.4f}')
print(f'  Log Loss:      {log_loss(y_true, y_proba):.4f}')
print(f'  Latência P50:  {np.percentile(latencies, 50):.2f} ms')
print(f'  Latência P95:  {np.percentile(latencies, 95):.2f} ms')
print(f'  Tempo treino:  {train_time/60:.1f} min')
print(f'{"="*55}')

print(f'\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['Legítimo', 'Phishing']))

print(f'Confusion Matrix:')
cm = confusion_matrix(y_true, y_pred)
print(f'  TN={cm[0][0]:,}  FP={cm[0][1]:,}')
print(f'  FN={cm[1][0]:,}  TP={cm[1][1]:,}')

In [ ]:
# ============================================================
# 14. Teste Rápido com URLs Conhecidas
# ============================================================
# Verifica se o modelo agora funciona com o formato completo

test_urls = [
    ('[URL] https://www.google.com [WHOIS] unknown [EXTRA] length=24 tls=1',       'LEGIT'),
    ('[URL] https://www.bb.com.br [WHOIS] unknown [EXTRA] length=22 tls=1',        'LEGIT'),
    ('[URL] https://github.com [WHOIS] unknown [EXTRA] length=18 tls=1',           'LEGIT'),
    ('[URL] http://192.168.1.1.login.xyz/steal [WHOIS] unknown [EXTRA] length=38 tls=0', 'PHISH'),
    ('[URL] http://bb-seguranca.ml/acesso [WHOIS] unknown [EXTRA] length=32 tls=0',     'PHISH'),
    ('[URL] http://paypal-verify.xyz/account [WHOIS] unknown [EXTRA] length=35 tls=0',  'PHISH'),
    # Também testar URL bruta para comparar
    ('https://www.google.com',                  'LEGIT'),
    ('http://192.168.1.1.login.xyz/steal',      'PHISH'),
]

model.eval()
print(f'{"Input":65s} | {"P(legit)":>8s} | {"P(phish)":>8s} | {"Pred":>8s} | {"Esp":>5s}')
print('-' * 105)

for text, expected in test_urls:
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=MAX_LENGTH, padding=True,
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
    p_legit = probs[0].item()
    p_phish = probs[1].item()
    pred = 'PHISH' if p_phish > 0.5 else 'LEGIT'
    ok = '✓' if pred == expected else '✗'
    display = text[:65]
    print(f'{display:65s} | {p_legit:>7.1%} | {p_phish:>7.1%} | {pred:>8s} | {ok:>5s}')

In [ ]:
# ============================================================
# 15. Salvar Modelo Final
# ============================================================
print(f'Salvando modelo em: {OUTPUT_DIR}')

# Salvar modelo + tokenizer (com os tokens especiais)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Salvar métricas
metrics = {
    'accuracy':     float(accuracy_score(y_true, y_pred)),
    'f1_score':     float(f1_score(y_true, y_pred)),
    'precision':    float(precision_score(y_true, y_pred)),
    'recall':       float(recall_score(y_true, y_pred)),
    'auc_roc':      float(roc_auc_score(y_true, y_proba)),
    'pr_auc':       float(average_precision_score(y_true, y_proba)),
    'mcc':          float(matthews_corrcoef(y_true, y_pred)),
    'log_loss':     float(log_loss(y_true, y_proba)),
    'latency_p50':  float(np.percentile(latencies, 50)),
    'latency_p95':  float(np.percentile(latencies, 95)),
    'train_time_s': float(train_time),
    'config': {
        'model_id':    MODEL_ID,
        'max_length':  MAX_LENGTH,
        'num_epochs':  NUM_EPOCHS,
        'batch_size':  BATCH_SIZE,
        'grad_accum':  GRAD_ACCUM,
        'lr':          LR,
        'special_tokens_added': SPECIAL_TOKENS,
        'train_samples': len(train_texts),
        'eval_samples':  len(eval_texts),
    }
}

with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nModelo salvo com sucesso!')
print(f'  {OUTPUT_DIR}/config.json')
print(f'  {OUTPUT_DIR}/model.safetensors')
print(f'  {OUTPUT_DIR}/tokenizer.json')
print(f'  {OUTPUT_DIR}/metrics.json')
print(f'\nPara usar na API, copie a pasta inteira para phishing-api/model/')

In [ ]:
# ============================================================
# 16. Verificação Final: tokens especiais no modelo salvo
# ============================================================
# Recarrega do disco para confirmar que está tudo certo
print('Recarregando modelo do disco para verificação...')

tok2 = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mod2 = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
mod2.to(device)
mod2.eval()

print(f'Vocab recarregado: {len(tok2):,}')
for token in SPECIAL_TOKENS:
    tid = tok2.convert_tokens_to_ids(token)
    unk = tok2.convert_tokens_to_ids('[UNK]')
    status = '✓ OK' if tid != unk else '✗ FALHOU'
    print(f'  {token:10s} -> ID={tid} {status}')

# Teste rápido com modelo recarregado
test = '[URL] https://www.google.com [WHOIS] unknown [EXTRA] length=24 tls=1'
inputs = tok2(test, return_tensors='pt', truncation=True, max_length=MAX_LENGTH, padding=True).to(device)
with torch.no_grad():
    probs = torch.softmax(mod2(**inputs).logits, dim=-1)[0]
print(f'\nTeste: {test}')
print(f'  P(legítimo) = {probs[0].item():.1%}')
print(f'  P(phishing) = {probs[1].item():.1%}')
print(f'  → {"LEGÍTIMO ✓" if probs[0] > probs[1] else "PHISHING ✗"}')

del tok2, mod2
gc.collect()
torch.cuda.empty_cache()
print('\nVerificação concluída!')

---
## Próximos passos

1. **Baixar o modelo** da pasta `TCC-Finetuning-DomURLs-BERT/modelo-final/` no Google Drive
2. **Copiar para** `phishing-api/model/` (substituindo o modelo antigo)
3. **Reverter** a mudança no `app.py` para voltar a usar `create_feature_text()` em vez de URL bruta
4. **Rebuildar o Docker**: `docker-compose build && docker-compose up -d`
5. **Validar**: `python whitelist/validar_modelo.py --api-url http://localhost:8000`

### Diferenças em relação ao treino anterior

| | Treino anterior | Este treino |
|---|---|---|
| Tokens especiais | `[UNK]` (não reconhecidos) | Adicionados ao vocabulário |
| `model.resize_token_embeddings()` | Não chamado | Chamado |
| Formato input | `[URL] url [WHOIS]... [EXTRA]...` | Mesmo (agora funciona) |
| `load_best_model_at_end` | False | True (melhor F1) |
| `save_strategy` | 'no' | 'epoch' (checkpoint por época) |